# 🧬 VIDEO-FREE · 6. Entrenar tu personaje (LoRA)
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivanherrerabaquero-dev/VIDEO-FREE/blob/main/colab/6_entrenar_personaje.ipynb)

Enseña a la IA **cómo es tu cara** (o la de tu actriz) para que el director (cuaderno 5) pueda crearlos en **cualquier pose y lugar**: de espaldas caminando al auto, de lado manejando, de lejos en la carretera. Todo sin perder el parecido.

- Usa los recortes de `medios/personajes/` y, para un parecido mucho mayor, **10 a 20 fotos reales** tuyas (cara y cuerpo, distintas luces y ángulos) que subas a tu Drive en `VIDEO-FREE/entrenamiento/actor/` (o `.../actriz/`).
- **Modelo base:** RealVisXL 5.0 (fotorrealista, OpenRAIL++). **Método:** DreamBooth LoRA con el script oficial de Diffusers (Apache 2.0), que se descarga al usarlo.
- **Tiempo:** ~40–70 min en T4 gratis (768 px); ~15 min en A100 (1024 px).
- Entrena **solo con tu cara o con la de alguien que te dé permiso**. Nada de caras de famosos.

**Antes de empezar (una sola vez):**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU.*
2. Crea un token de GitHub (*Settings → Developer settings → Fine-grained tokens*) con acceso solo a **VIDEO-FREE** y permiso **Contents: Read and write**.
3. En Colab, abre el icono de **llave (Secretos)**, crea `GITHUB_TOKEN`, pega el token y activa *Acceso al cuaderno*.



In [ ]:
#@title 1 · Preparar Colab y conectar con tu repo
REPO = "ivanherrerabaquero-dev/VIDEO-FREE"  #@param {type:"string"}
RAMA = "main"  #@param {type:"string"}
#@markdown Guarda copias en tu Google Drive (Colab te pedirá permiso para acceder a Drive).
USAR_DRIVE = True  #@param {type:"boolean"}
import os, sys, subprocess
if os.path.exists("/content/VIDEO-FREE/.git"):
    subprocess.run(["git", "-C", "/content/VIDEO-FREE", "pull", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, f"https://github.com/{REPO}", "/content/VIDEO-FREE"], check=True)
sys.path.insert(0, "/content/VIDEO-FREE/colab")
import importlib, comun; importlib.reload(comun)
info = comun.preparar(REPO, RAMA, usar_drive=USAR_DRIVE)
VRAM = info["gpu"].get("vram_gb", 0)

In [ ]:
#@title 2 · Instalar (2–3 min)
!pip install -q -U "diffusers>=0.35" transformers accelerate peft bitsandbytes datasets
import diffusers, urllib.request, importlib, director
importlib.reload(director)
version = diffusers.__version__.split(".dev")[0]
urllib.request.urlretrieve(f"https://raw.githubusercontent.com/huggingface/diffusers/v{version}/examples/dreambooth/train_dreambooth_lora_sdxl.py",
                           "/content/entrenar_lora_sdxl.py")
print("Diffusers", version, "· script de entrenamiento listo")

In [ ]:
#@title 3 · Preparar las fotos
PERSONAJE = "actor"  #@param ["actor", "actriz"]
PALABRA = ""  #@param {type:"string"}
#@markdown Palabra clave del personaje. Vacío = `jsnv man` para el actor y `mrla woman` para la actriz.
import glob, json, shutil
from pathlib import Path
from PIL import Image, ImageOps
from IPython.display import display
PALABRA = PALABRA.strip() or ("jsnv man" if PERSONAJE == "actor" else "mrla woman")
RES = 1024 if VRAM >= 22 else 768
DATOS = Path(f"/content/datos_{PERSONAJE}")
shutil.rmtree(DATOS, ignore_errors=True); DATOS.mkdir(parents=True)
FONDOS = [(126, 126, 126), (212, 188, 158), (58, 68, 88), (232, 232, 226), (148, 108, 78)]

def sobre_fondo(rgba, color):
    fondo = Image.new("RGB", rgba.size, color)
    sombra = Image.new("RGB", rgba.size, tuple(max(0, c - 55) for c in color))
    fondo = Image.composite(sombra, fondo, Image.linear_gradient("L").resize(rgba.size).point(lambda v: int(v * 0.4)))
    fondo.paste(rgba, (0, 0), rgba)
    return fondo

def a_cuadrado(img, color):
    s = max(img.size)
    lienzo = Image.new("RGB", (s, s), color)
    lienzo.paste(img, ((s - img.width) // 2, (s - img.height) // 2))
    return lienzo.resize((RES, RES), Image.LANCZOS)

n = 0
def guardar_foto(img):
    global n
    for espejo in (False, True):
        (ImageOps.mirror(img) if espejo else img).save(DATOS / f"{n:03d}.png"); n += 1

recortes = sorted(glob.glob(f"/content/VIDEO-FREE/medios/personajes/{PERSONAJE}_*.png"))
for k, ruta in enumerate(recortes):
    rgba = Image.open(ruta).convert("RGBA")
    rgba = rgba.crop(rgba.getchannel("A").point(lambda v: 255 if v > 8 else 0).getbbox())
    for j in range(2):
        color = FONDOS[(k + j * 2) % len(FONDOS)]
        guardar_foto(a_cuadrado(sobre_fondo(rgba, color), color))
        if rgba.height > rgba.width * 1.3:  # foto de cuerpo entero: también un primer plano de la cara
            cara = rgba.crop((0, 0, rgba.width, min(rgba.height, rgba.width)))
            guardar_foto(a_cuadrado(sobre_fondo(cara, color), color))
propias = []
if comun.en_drive(f"entrenamiento/{PERSONAJE}"):
    propias = [f for f in sorted(comun.en_drive(f"entrenamiento/{PERSONAJE}").iterdir()) if f.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp")]
for ruta in propias:
    img = ImageOps.exif_transpose(Image.open(ruta)).convert("RGB")
    guardar_foto(ImageOps.fit(img, (RES, RES), Image.LANCZOS, centering=(0.5, 0.35)))
print(f"{len(recortes)} recortes del repo + {len(propias)} fotos tuyas → {n} imágenes de entrenamiento a {RES} px")
if len(propias) < 8:
    print("Consejo: con 10–20 fotos reales tuyas en Drive (VIDEO-FREE/entrenamiento/%s/) el parecido mejora mucho." % PERSONAJE)
display(Image.open(director.hoja_de_contactos(sorted(DATOS.glob("*.png"))[::2][:12], "/content/datos.jpg", columnas=6, ancho=180)))

In [ ]:
#@title 4 · Entrenar
PASOS = 800  #@param {type:"slider", min:200, max:2000, step:100}
RANGO = 16  #@param [8, 16, 32] {type:"raw"}
#@markdown Más rango = aprende más detalle (y el archivo pesa más).
APRENDIZAJE = 1e-4  #@param {type:"number"}
SALIDA = Path(f"/content/lora_{PERSONAJE}")
comun.sh(f'python /content/entrenar_lora_sdxl.py --pretrained_model_name_or_path SG161222/RealVisXL_V5.0 --variant fp16 '
         f'--instance_data_dir "{DATOS}" --instance_prompt "a photo of {PALABRA}" --output_dir "{SALIDA}" '
         f'--resolution {RES} --train_batch_size 1 --gradient_accumulation_steps 1 --gradient_checkpointing --use_8bit_adam '
         f'--mixed_precision fp16 --learning_rate {APRENDIZAJE} --lr_scheduler constant --lr_warmup_steps 0 '
         f'--max_train_steps {PASOS} --rank {RANGO} --seed 0 --checkpointing_steps 100000 --report_to tensorboard')
print("Entrenamiento terminado:", SALIDA / "pytorch_lora_weights.safetensors")

In [ ]:
#@title 5 · Probar el personaje entrenado
import gc, torch
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
pipe = StableDiffusionXLPipeline.from_pretrained("SG161222/RealVisXL_V5.0", torch_dtype=torch.float16, variant="fp16").to("cuda")
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True, algorithm_type="sde-dpmsolver++")
pipe.load_lora_weights(str(SALIDA), weight_name="pytorch_lora_weights.safetensors")
pruebas = [
    f"photo of {PALABRA} driving a black 1969 convertible on a desert highway at sunset, wind in the hair",
    f"photo of {PALABRA} walking toward a parked black 1969 convertible, seen from behind, golden hour",
    f"close-up portrait of {PALABRA} singing with emotion, night, dashboard light",
    f"photo of {PALABRA} standing in a field of tall dry grass, overcast, melancholic",
]
fotos = []
for k, pr in enumerate(pruebas):
    img = pipe(prompt=f"{pr}, {director.ESTILO_FOTO}", negative_prompt=director.NEGATIVO_FOTO, width=1152, height=896,
               num_inference_steps=30, guidance_scale=5.0, generator=torch.Generator("cuda").manual_seed(k)).images[0]
    f = comun.TRABAJO / f"prueba_{PERSONAJE}_{k}.jpg"; img.save(f); fotos.append(f)
display(Image.open(director.hoja_de_contactos(fotos, comun.TRABAJO / "pruebas.jpg", columnas=2, ancho=560)))
del pipe; gc.collect(); torch.cuda.empty_cache()
print("Si no se parece lo suficiente: agrega fotos reales tuyas en Drive o sube los PASOS, y entrena otra vez.")

In [ ]:
#@title 6 · Guardar el personaje entrenado
archivo = SALIDA / "pytorch_lora_weights.safetensors"
comun.a_drive(archivo, f"lora/{PERSONAJE}.safetensors")
mb = archivo.stat().st_size / 2**20
if mb < 95:
    comun.guardar(f"medios/lora/{PERSONAJE}.safetensors", archivo.read_bytes(), f"Personaje entrenado: {PERSONAJE}")
else:
    print(f"El archivo pesa {mb:.0f} MB: queda solo en tu Drive (GitHub no acepta más de 100 MB).")
info = {"palabra": PALABRA, "base": "SG161222/RealVisXL_V5.0", "pasos": PASOS, "rango": RANGO, "resolucion": RES, "imagenes": n}
comun.guardar(f"medios/lora/{PERSONAJE}.json", json.dumps(info, ensure_ascii=False, indent=1), f"Personaje entrenado: {PERSONAJE} (datos)")
print("Listo. En el cuaderno 5 elige MODO_PERSONAJE = «entrenado (pose libre)».")